# Robo-Greeno — (3+3) Hexapod gait in Colab

A six-legged spider walker with a **tripod gait**: three legs in stance while the other three swing. Builds directly on the single-leg kinematics notebook — each leg here is essentially the same hip/knee/ankle chain, just six of them attached to a chassis.

What the notebook produces:
1. A MuJoCo model of a hexapod (1 chassis + 6 legs, 3 joints per leg = 24 DOF).
2. A CPG (Central Pattern Generator) that drives the joints with a tripod gait.
3. A rendered side-view video of the spider walking forward.
4. **Three diagnostic plots**: gait diagram (which legs are in stance over time), foot height per leg, and chassis trajectory.

This is **kinematic animation** — we drive every joint and the chassis pose directly each frame and call `mj_forward`. There's no physics integration, no contact forces. That keeps the notebook reliable across machines and lets you focus on gait analysis. The 'next steps' cell at the bottom shows what to change for real physics.

In [ ]:
# Colab setup. Run this first.
import os, sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "mujoco", "mediapy", "matplotlib"])

os.environ.setdefault("MUJOCO_GL", "egl")

import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import mujoco

print("MuJoCo:", mujoco.__version__)

In [ ]:
# Build the hexapod XML programmatically. Each leg is the same template; we just
# attach 6 copies at the right (x, y) positions on the chassis with the correct
# axis signs so positive joint angles mean the same thing (forward, lift, bend)
# for both left and right sides.

CHASSIS_LX, CHASSIS_LY, CHASSIS_LZ = 0.16, 0.10, 0.025   # half-extents
COXA_LEN  = 0.05
FEMUR_LEN = 0.15
TIBIA_LEN = 0.20
BODY_Z    = 0.22                                          # standing height

# 6 legs: 3 on each side. (x = fore-aft attachment, side = +y left / -y right)
LEG_LAYOUT = [
    # name,        attach_x,  side ('L'/'R')
    ("LF",          0.12,    'L'),    # left  front
    ("LM",          0.00,    'L'),    # left  middle
    ("LR",         -0.12,    'L'),    # left  rear
    ("RF",          0.12,    'R'),    # right front
    ("RM",          0.00,    'R'),    # right middle
    ("RR",         -0.12,    'R'),    # right rear
]

def make_leg(name, attach_x, side):
    """Return the XML for one leg attached to the chassis.

    Convention: positive coxa  = leg swings toward +x (forward)
                positive femur = leg lifts up (foot toward +z)
                positive tibia = knee bends (foot tucks under)
    Left/right symmetry is achieved by flipping the axis signs on the right side.
    """
    if side == 'L':
        attach_y = +CHASSIS_LY
        leg_y_sign = +1                              # leg extends in +y
        coxa_axis  = "0 0 -1"                        # +coxa = forward (left side)
        lift_axis  = "1 0 0"                         # +femur = lift up (left side)
    else:
        attach_y = -CHASSIS_LY
        leg_y_sign = -1                              # leg extends in -y
        coxa_axis  = "0 0 1"                         # +coxa = forward (right side)
        lift_axis  = "-1 0 0"                        # +femur = lift up (right side)

    fy = leg_y_sign
    return f"""
      <body name=\"{name}_coxa\" pos=\"{attach_x} {attach_y} 0\">
        <joint name=\"{name}_coxa_j\"  axis=\"{coxa_axis}\" range=\"-45 45\"/>
        <geom  name=\"{name}_coxa_g\"  type=\"capsule\" size=\"0.012\"
               fromto=\"0 0 0  0 {fy*COXA_LEN} 0\" rgba=\"0.30 0.50 0.30 1\"/>
        <body name=\"{name}_femur\" pos=\"0 {fy*COXA_LEN} 0\">
          <joint name=\"{name}_femur_j\" axis=\"{lift_axis}\" range=\"-60 60\"/>
          <geom  name=\"{name}_femur_g\" type=\"capsule\" size=\"0.011\"
                 fromto=\"0 0 0  0 {fy*FEMUR_LEN} 0\" rgba=\"0.20 0.65 0.40 1\"/>
          <body name=\"{name}_tibia\" pos=\"0 {fy*FEMUR_LEN} 0\">
            <joint name=\"{name}_tibia_j\" axis=\"{lift_axis}\" range=\"-30 90\"/>
            <geom  name=\"{name}_tibia_g\" type=\"capsule\" size=\"0.009\"
                   fromto=\"0 0 0  0 {fy*TIBIA_LEN} 0\" rgba=\"0.10 0.75 0.45 1\"/>
            <site name=\"{name}_foot\" pos=\"0 {fy*TIBIA_LEN} 0\"
                  size=\"0.012\" rgba=\"1 0.4 0 1\"/>
          </body>
        </body>
      </body>"""

legs_xml = "\n".join(make_leg(name, x, s) for name, x, s in LEG_LAYOUT)

MODEL_XML = f"""
<mujoco model=\"robo_greeno_3plus3\">
  <compiler angle=\"degree\" autolimits=\"true\"/>
  <option gravity=\"0 0 -9.81\" timestep=\"0.005\"/>
  <default>
    <joint type=\"hinge\" damping=\"0.1\"/>
    <geom  contype=\"0\" conaffinity=\"0\" density=\"500\"/>
  </default>

  <worldbody>
    <light pos=\"0 -2 2\" dir=\"0 0.5 -1\"/>
    <camera name=\"chase\"  pos=\"-0.6 -0.9 0.55\" xyaxes=\"0.83 -0.55 0  0.18 0.27 0.95\" fovy=\"55\"/>
    <camera name=\"side\"   pos=\"0.0 -1.2 0.35\"  xyaxes=\"1 0 0  0 0.3 1\"             fovy=\"50\"/>
    <camera name=\"top\"    pos=\"0.0  0.0 1.2\"   xyaxes=\"1 0 0  0 1 0\"               fovy=\"55\"/>
    <geom name=\"ground\" type=\"plane\" size=\"3 3 0.05\" rgba=\"0.88 0.88 0.92 1\"/>

    <body name=\"chassis\" pos=\"0 0 {BODY_Z}\">
      <freejoint name=\"root\"/>
      <geom name=\"chassis_g\" type=\"box\"
            size=\"{CHASSIS_LX} {CHASSIS_LY} {CHASSIS_LZ}\"
            rgba=\"0.20 0.50 0.30 1\"/>
      <site name=\"chassis_site\" pos=\"0 0 0\" size=\"0.02\" rgba=\"1 1 0 1\"/>
      {legs_xml}
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MODEL_XML)
data  = mujoco.MjData(model)

print("nq (qpos size):", model.nq)
print("nv (qvel size):", model.nv)
print("number of bodies:", model.nbody, " → chassis + 6 × (coxa+femur+tibia) = 19 bodies + world")

In [ ]:
# Helper indices so we can write joints/sites by leg name.
LEG_NAMES = [name for name, _, _ in LEG_LAYOUT]

joint_qpos_addr = {}
for name in LEG_NAMES:
    for j in ("coxa", "femur", "tibia"):
        jname = f"{name}_{j}_j"
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jname)
        joint_qpos_addr[(name, j)] = model.jnt_qposadr[jid]

foot_site_id = {
    name: mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, f"{name}_foot")
    for name in LEG_NAMES
}

chassis_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "chassis")
ROOT_QPOS = 0  # free joint occupies qpos[0:7] = (x, y, z, qw, qx, qy, qz)

# Sanity check: render the rest pose (qpos = 0 everywhere except chassis height).
def render_pose(camera_name="chase", width=720, height=480):
    with mujoco.Renderer(model, height=height, width=width) as r:
        r.update_scene(data, camera_name)
        return r.render().copy()

data.qpos[:] = 0
data.qpos[ROOT_QPOS + 2] = BODY_Z       # z
data.qpos[ROOT_QPOS + 3] = 1.0           # qw = 1, rest of quat = 0
# Tuck the legs into a standing pose: femur down 20°, tibia bent 60°
for name in LEG_NAMES:
    data.qpos[joint_qpos_addr[(name, "femur")]] = np.deg2rad(-20)
    data.qpos[joint_qpos_addr[(name, "tibia")]] = np.deg2rad(-60)
mujoco.mj_forward(model, data)

media.show_image(render_pose())

## The tripod CPG

Each leg has its own phase variable that runs from 0 to 1 and resets. **Phase < 0.5 is stance** (foot planted, leg sweeps back); **phase ≥ 0.5 is swing** (femur lifts, leg returns forward).

The 3+3 tripod gait is just two groups of legs offset by 0.5:
- **Tripod A**: LF, RM, LR — phase offset 0
- **Tripod B**: RF, LM, RR — phase offset 0.5

Diagonal opposites move together; at any moment three legs (one front + one mid + one rear from opposite sides) support the body.

In [ ]:
# Tripod assignment (group 0 = tripod A, group 1 = tripod B).
TRIPOD_GROUP = {"LF": 0, "RM": 0, "LR": 0, "RF": 1, "LM": 1, "RR": 1}

def cpg_joints(t, leg_name,
               period=0.7, coxa_amp=22.0,
               femur_base=-20.0, femur_lift=30.0,
               tibia_base=-60.0):
    """Return (coxa, femur, tibia) target angles in DEGREES for one leg at time t."""
    offset = 0.5 if TRIPOD_GROUP[leg_name] == 1 else 0.0
    phase = ((t / period) + offset) % 1.0

    # Coxa: cosine swing.  +amp at phase 0 (forward), -amp at phase 0.5 (back).
    coxa = coxa_amp * np.cos(2 * np.pi * phase)

    # Femur: stay at base during stance, lift in an arc during swing.
    if phase < 0.5:                       # stance
        femur = femur_base
    else:                                  # swing
        s = (phase - 0.5) / 0.5
        femur = femur_base + femur_lift * np.sin(np.pi * s)

    # Tibia: keep the bend constant for simplicity.
    tibia = tibia_base
    return coxa, femur, tibia, phase

In [ ]:
# Animate: drive the chassis forward at constant velocity, set joint angles from
# the CPG, and call mj_forward each frame.  Record state for analysis.

DURATION   = 4.0    # seconds
FPS        = 30
DT_RECORD  = 1.0 / FPS
BODY_VEL   = 0.18   # m/s forward in +x

n_frames = int(DURATION * FPS)
times    = np.arange(n_frames) * DT_RECORD

frames    = []
foot_pos  = {name: np.zeros((n_frames, 3)) for name in LEG_NAMES}
foot_phase= {name: np.zeros(n_frames) for name in LEG_NAMES}
chassis_xyz = np.zeros((n_frames, 3))

with mujoco.Renderer(model, height=420, width=720) as renderer:
    for k, t in enumerate(times):
        # 1. Chassis advances forward in world frame.
        data.qpos[ROOT_QPOS + 0] = BODY_VEL * t
        data.qpos[ROOT_QPOS + 1] = 0.0
        data.qpos[ROOT_QPOS + 2] = BODY_Z
        data.qpos[ROOT_QPOS + 3] = 1.0
        data.qpos[ROOT_QPOS + 4:ROOT_QPOS + 7] = 0.0

        # 2. CPG sets each leg's joint angles.
        for name in LEG_NAMES:
            c, f, tib, ph = cpg_joints(t, name)
            data.qpos[joint_qpos_addr[(name, "coxa")]]  = np.deg2rad(c)
            data.qpos[joint_qpos_addr[(name, "femur")]] = np.deg2rad(f)
            data.qpos[joint_qpos_addr[(name, "tibia")]] = np.deg2rad(tib)
            foot_phase[name][k] = ph

        # 3. Forward kinematics propagates qpos → world-space site positions.
        mujoco.mj_forward(model, data)

        # 4. Record.
        chassis_xyz[k] = data.qpos[ROOT_QPOS:ROOT_QPOS + 3]
        for name in LEG_NAMES:
            foot_pos[name][k] = data.site_xpos[foot_site_id[name]]

        # 5. Render.
        renderer.update_scene(data, "chase")
        frames.append(renderer.render().copy())

print(f"recorded {n_frames} frames over {DURATION:.1f} s")
print(f"chassis traveled {chassis_xyz[-1,0]-chassis_xyz[0,0]:.3f} m in x")

In [ ]:
# Watch it walk.
media.show_video(frames, fps=FPS)

## Gait diagram

The standard biomechanics plot: one row per leg, shaded where the leg is in **stance** (phase < 0.5), blank where it's in **swing**. A clean tripod gait should show two interleaved bands: the three legs of tripod A in stance together while tripod B is in swing, then they swap, perfectly out of phase.

In [ ]:
# Stance = phase < 0.5
stance = {name: foot_phase[name] < 0.5 for name in LEG_NAMES}

fig, ax = plt.subplots(figsize=(9, 3.2))
for i, name in enumerate(LEG_NAMES):
    color = "#2c8f50" if TRIPOD_GROUP[name] == 0 else "#a26a2c"
    in_stance = stance[name]
    # Draw horizontal bars where in_stance is True.
    starts = np.where(np.diff(in_stance.astype(int)) == 1)[0]
    ends   = np.where(np.diff(in_stance.astype(int)) == -1)[0]
    if in_stance[0]:
        starts = np.concatenate(([0], starts))
    if in_stance[-1]:
        ends = np.concatenate((ends, [len(in_stance) - 1]))
    for s, e in zip(starts, ends):
        ax.barh(i, times[e] - times[s], left=times[s], height=0.7, color=color)

ax.set_yticks(range(len(LEG_NAMES)))
ax.set_yticklabels(LEG_NAMES)
ax.invert_yaxis()
ax.set_xlabel("time (s)")
ax.set_xlim(0, DURATION)
ax.set_title("Gait diagram — shaded = stance (foot on ground)")
ax.grid(axis="x", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Foot height per leg

Foot z-coordinate over time. Each leg's foot should sit near z = 0 during stance and rise into an arc during swing. The two tripods’ arcs should be perfectly offset.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
for name in LEG_NAMES:
    color = "#2c8f50" if TRIPOD_GROUP[name] == 0 else "#a26a2c"
    style = "-" if name.startswith("L") else "--"
    ax.plot(times, foot_pos[name][:, 2], style, color=color, label=name, linewidth=1.6, alpha=0.9)

ax.set_xlabel("time (s)")
ax.set_ylabel("foot z position (m)")
ax.set_title("Foot height vs time — arcs are the swing phase")
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.2))
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

## Chassis trajectory and foot prints (top view)

Top-down view: chassis trajectory (the line) plus where each foot touched the ground (dots). For a clean tripod gait, the dots form two staggered rows along the direction of travel.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))

# Chassis trajectory
ax.plot(chassis_xyz[:, 0], chassis_xyz[:, 1], "-", color="#1f5fa8", linewidth=2.0, label="chassis path")

# Footprints: scatter foot (x, y) only at samples where the leg is in stance.
for name in LEG_NAMES:
    mask = stance[name]
    color = "#2c8f50" if TRIPOD_GROUP[name] == 0 else "#a26a2c"
    marker = "o" if name.startswith("L") else "s"
    ax.scatter(foot_pos[name][mask, 0], foot_pos[name][mask, 1],
               c=color, marker=marker, s=22, alpha=0.55,
               edgecolor="white", linewidth=0.5, label=name)

ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("x (m, walking direction)")
ax.set_ylabel("y (m)")
ax.set_title("Top view — chassis path (blue) and footprints during stance")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=9)
plt.tight_layout()
plt.show()

## Next steps

This notebook is **kinematic** — every joint and the chassis pose are scripted, then `mj_forward` propagates them to world coordinates. To turn it into a real physics simulation of a walking spider, you change four things:

1. **Turn collisions on.** In the `<default>` block, change `contype="0" conaffinity="0"` to `contype="1" conaffinity="1"`. Foot capsules and the ground plane will then interact.
2. **Add position actuators.** Inside the `<actuator>` block (add one), declare a `<position name="..." joint="..." kp="30"/>` for each of the 18 leg joints. The CPG no longer writes `data.qpos`; it writes `data.ctrl`.
3. **Replace `mj_forward` with `mj_step`** in the loop. Time integrates, contacts apply forces, the chassis rises/falls in response to leg dynamics.
4. **Tune `kp`, `damping`, `friction`** until the spider stays upright. Start with `kp=30`, joint `damping=0.5`, foot `friction="1.5 0.005 0.0001"`. Expect a few iterations.

Once those four changes are in, every analysis cell here keeps working unchanged — the gait diagram, foot heights, and chassis trajectory all read the same arrays.

Beyond that, the natural extensions are: **wave gait** (`TRIPOD_GROUP` becomes a 6-way phase split with 1/6 offsets), **turning** (per-side velocity differential), **rough terrain** (replace the ground plane with a heightfield), and **learned control** (swap the CPG for a small neural net trained with RL via MJX).